# 自我修订与CRITIC：迭代输出提升

自我修订将LLM用作三种用途————生成、反馈、修订，在一个循环中。CRITIC 将反馈这一步加固为外部工具调用的验证。

## 问题描述

一个agent产生了看起来正确的答案。可能只有一行代码存在语法错误，可能总结得太长，或者一个计划没有考虑到某个边界场景。你想要的是：agent能够批判它自己的输出，然后修好它。

自我修订使用单一模型验证了这一想法，没有训练数据，没有强化学习。但是这里有一个需要注意的地方，LLM对于硬性事实的自我验证能力很弱。CRITIC给出了修法————将验证步骤交给外部工具（搜索、代码解释器、计算器、测试机）。

这两篇论文共同组成了2026年迭代输出提升的默认：生成、验证（可以的话外部）、修订，当验证器通过的时候停止。

## Self-Refine 与 Reflexion 异同

两者都是「用自然语言代替梯度」的自我改进，但改的对象和记忆跨度不同。

### 相同点
- 都不更新模型权重，靠提示词里的自然语言反馈驱动下一轮。
- 都有「产出 → 评价/反馈 → 再产出」的循环，可设最大轮次与停止条件。
- 都假设：模型（或外部信号）能指出问题，且修正后通常会更好。

### 不同点

| | **Self-Refine** | **Reflexion** |
|---|---|---|
| 核心循环 | 生成 → **反馈** → **修订同一份输出** | Actor 轨迹 → **评估** → **反思写入情景记忆** → **重跑整次尝试** |
| 改进对象 | 当前答案/草稿本身（同一任务实例上打磨） | 下一次尝试的策略/行为（跨尝试学习） |
| 反馈形态 | 针对输出的具体批注（哪里不好、怎么改） | 针对失败轨迹的教训（错用了什么工具、误解了什么） |
| 记忆 | 多为本轮对话里的草稿+反馈链 | **情景记忆**：反思列表前缀到后续尝试 |
| 典型信号 | 自评质量、风格、完整性；CRITIC 则接外部工具验事实 | 标量（对错/测试）、启发式（卡住）、或自我评估 |
| 何时更合适 | 作文、代码草稿、计划文本等「同一结果要改到过关」 | Agent 多步工具任务失败后「换一条路再试」 |

一句话：**Self-Refine 是改答案；Reflexion 是改下一次怎么做。** CRITIC 可视为 Self-Refine 族里把「反馈」换成外部验证器的加强版。
